# Day 6 — SQL: GROUP BY, GROUPING SETS, ROLLUP

> **Topics:** GROUP BY · ROLLUP · GROUPING SETS · CUBE · GROUPING() function  
> **Run order:** top to bottom — Cell 2 connects and creates all tables inline

In [ ]:
%load_ext sql

USERNAME = 'postgres'
PASSWORD = 'hariom'
HOST     = 'localhost'
PORT     = '5432'
DBNAME   = 'postgres'

%sql postgresql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}

In [ ]:
%%sql
DROP TABLE IF EXISTS d6_sales CASCADE;

CREATE TABLE d6_sales (
    sale_id   SERIAL PRIMARY KEY,
    region    VARCHAR(20),
    category  VARCHAR(20),
    product   VARCHAR(30),
    sale_date DATE,
    revenue   NUMERIC(10, 2)
);

INSERT INTO d6_sales (region, category, product, sale_date, revenue) VALUES
  ('North', 'Electronics', 'Laptop',   '2024-01-05', 1200.00),
  ('North', 'Electronics', 'Phone',    '2024-01-10',  800.00),
  ('North', 'Furniture',   'Chair',    '2024-01-12',  300.00),
  ('North', 'Furniture',   'Desk',     '2024-01-15',  600.00),
  ('South', 'Electronics', 'Tablet',   '2024-01-08',  500.00),
  ('South', 'Electronics', 'Laptop',   '2024-01-20', 1100.00),
  ('South', 'Furniture',   'Sofa',     '2024-01-18',  900.00),
  ('South', 'Furniture',   'Wardrobe', '2024-01-22', 1500.00),
  ('East',  'Electronics', 'Phone',    '2024-01-09',  750.00),
  ('East',  'Furniture',   'Chair',    '2024-01-14',  280.00);

In [ ]:
%%sql
SELECT * FROM d6_sales ORDER BY region, category, sale_date;

---
## 1. Basic GROUP BY — One Row per (Region, Category)

In [ ]:
%%sql
SELECT   region,
         category,
         COUNT(*)            AS num_sales,
         SUM(revenue)        AS total_revenue,
         ROUND(AVG(revenue), 2) AS avg_revenue
FROM     d6_sales
GROUP BY region, category
ORDER BY region, category;

---
## 2. ROLLUP — Hierarchical Subtotals

`ROLLUP(region, category)` produces:
1. (region, category) — detail rows
2. (region, NULL)     — subtotal per region
3. (NULL, NULL)       — grand total

**NULL in the output = that column was rolled up (aggregated away)**

In [ ]:
%%sql
-- Raw ROLLUP — NULLs show which rows are subtotals
SELECT   region,
         category,
         SUM(revenue) AS total_revenue
FROM     d6_sales
GROUP BY ROLLUP(region, category)
ORDER BY region NULLS LAST, category NULLS LAST;

In [ ]:
%%sql
-- ROLLUP with labeled subtotal rows using COALESCE
SELECT
    COALESCE(region,   'ALL REGIONS')    AS region,
    COALESCE(category, 'ALL CATEGORIES') AS category,
    SUM(revenue)                         AS total_revenue
FROM   d6_sales
GROUP BY ROLLUP(region, category)
ORDER BY region NULLS LAST, category NULLS LAST;

In [ ]:
%%sql
-- GROUPING() function: 1 = rolled up, 0 = real value
-- Use when your data might actually contain NULL values (distinguishes data-NULL from rollup-NULL)
SELECT
    CASE WHEN GROUPING(region)   = 1 THEN 'ALL REGIONS'    ELSE region   END AS region,
    CASE WHEN GROUPING(category) = 1 THEN 'ALL CATEGORIES' ELSE category END AS category,
    SUM(revenue)          AS total_revenue,
    GROUPING(region)      AS region_rolled,
    GROUPING(category)    AS cat_rolled
FROM   d6_sales
GROUP BY ROLLUP(region, category)
ORDER BY region_rolled, cat_rolled, region NULLS LAST, category NULLS LAST;

---
## 3. GROUPING SETS — Custom Level Combinations

Pick exactly which grouping levels you need. No unwanted extras.

- `ROLLUP(a, b)` ≡ `GROUPING SETS((a,b),(a),())`
- Use `GROUPING SETS` when you need cross-dimension subtotals (both region-only AND category-only)

In [ ]:
%%sql
SELECT
    COALESCE(region,   'ALL')  AS region,
    COALESCE(category, 'ALL')  AS category,
    SUM(revenue)               AS total_revenue
FROM   d6_sales
GROUP  BY GROUPING SETS (
    (region, category),   -- detail: one row per region+category
    (region),             -- region subtotal
    (category),           -- category subtotal (ROLLUP alone wouldn't give this)
    ()                    -- grand total
)
ORDER BY region NULLS LAST, category NULLS LAST;

In [ ]:
%%sql
-- Label the grouping level for clarity
SELECT
    COALESCE(region,   'ALL')  AS region,
    COALESCE(category, 'ALL')  AS category,
    SUM(revenue)               AS total_revenue,
    CASE
        WHEN GROUPING(region) = 0 AND GROUPING(category) = 0 THEN 'region+category'
        WHEN GROUPING(region) = 0                             THEN 'region only'
        WHEN GROUPING(category) = 0                           THEN 'category only'
        ELSE 'grand total'
    END AS grouping_level
FROM   d6_sales
GROUP  BY GROUPING SETS (
    (region, category),
    (region),
    (category),
    ()
)
ORDER BY grouping_level, region NULLS LAST, category NULLS LAST;

---
## 4. CUBE — All Possible Combinations

`CUBE(a, b)` ≡ `GROUPING SETS((a,b),(a),(b),())`

Same result as the GROUPING SETS above (when you list all 4 combinations).

In [ ]:
%%sql
SELECT
    COALESCE(region,   'ALL') AS region,
    COALESCE(category, 'ALL') AS category,
    SUM(revenue)              AS total_revenue
FROM   d6_sales
GROUP  BY CUBE(region, category)
ORDER  BY region NULLS LAST, category NULLS LAST;

---
## Quick Reference

| Feature | Syntax | Produces |
|---------|--------|----------|
| Basic GROUP BY | `GROUP BY a, b` | One row per (a, b) |
| ROLLUP | `GROUP BY ROLLUP(a, b)` | (a,b) + (a) + () |
| GROUPING SETS | `GROUP BY GROUPING SETS((a,b),(a),(b),())` | Custom levels |
| CUBE | `GROUP BY CUBE(a, b)` | All 4 combinations |
| Label NULL | `COALESCE(col, 'ALL')` | Replace rollup-NULL |
| Detect level | `GROUPING(col)` = 1 → rolled up | 0 = real value |